In [1]:
import pandas as pd
from tqdm import tqdm

tqdm.pandas()

# Собираем данные о всех станциях

In [2]:
before_2020 = pd.read_parquet('..//data//before_2020.parquet')
after_2020 = pd.read_parquet('..//data//after_2020.parquet')

In [3]:
tmp = after_2020.dropna(subset=['start_station_name', 'end_station_name'])

start = tmp.groupby('start_station_name', as_index=False).agg({
    'start_lat': lambda x: x.mode()[0] if not x.mode().empty else None,
    'start_lng': lambda x: x.mode()[0] if not x.mode().empty else None
})
start.columns = ['station_name', 'lat', 'lng']

end = tmp.groupby('end_station_name', as_index=False).agg({
    'end_lat': lambda x: x.mode()[0] if not x.mode().empty else None,
    'end_lng': lambda x: x.mode()[0] if not x.mode().empty else None
})
end.columns = ['station_name', 'lat', 'lng']

stations = pd.concat([start, end]).drop_duplicates('station_name').reset_index(drop=True)

In [4]:
stations_csv = pd.read_csv('..//data//stations.csv')
stations_csv = stations_csv.drop(columns=['id', 'dpcapacity', 'online_date', 'city'])
stations_csv = stations_csv.rename({'name': 'station_name', 'latitude': 'lat', 'longitude': 'lng'}, axis=1)
stations_csv = stations_csv.drop_duplicates()

In [5]:
stations = pd.concat([stations, stations_csv]).drop_duplicates('station_name').reset_index(drop=True)

In [6]:
before_2020['user_type'] = before_2020['user_type'].map({'Subscriber': 'member', 'Customer': 'casual'})
before_2020 = before_2020.dropna(subset=['user_type'])

before_2020 = before_2020.drop(columns=['bike_id', 'start_station_id', 'end_station_id', 'ride_id'])

before_2020['trip_duration'] = before_2020['trip_duration'].dt.seconds
before_2020 = before_2020.sort_values('started_at')

In [7]:
before_2020['age'] = before_2020.started_at.dt.year - before_2020['birth_year']
before_2020 = before_2020.drop(columns=['birth_year'])

before_2020['age'] = before_2020['age'].fillna(before_2020['age'].mean())

In [8]:
after_2020 = after_2020.drop(columns=['start_station_id', 'end_station_id', 'ride_id'])

after_2020['trip_duration'] = after_2020['trip_duration'].dt.seconds
after_2020 = after_2020.sort_values('started_at')

В датасете много безымянных станций, чьи координаы неоднократно повторяются. Вынесем их также

## Выносим nan-станции

In [9]:
unknown_counter = 0
station_map = {}

def start_fill_with_unique_id(x):
    global unknown_counter
    if pd.isna(x['start_station_name']):
        coords = (x['start_lat'], x['start_lng'])
        if coords in station_map:
            return station_map[coords]
        else:
            val = f"unknown_station{unknown_counter}"
            station_map[coords] = val
            unknown_counter += 1
            return val
    return x['start_station_name']

def end_fill_with_unique_id(x):
    global unknown_counter
    if pd.isna(x['end_station_name']) and not pd.isna(x['end_lat']) and not pd.isna(x['end_lng']):
        coords = (x['end_lat'], x['end_lng'])
        if coords in station_map:
            return station_map[coords]
        else:
            val = f"unknown_station{unknown_counter}"
            station_map[coords] = val
            unknown_counter += 1
            return val
    return x['end_station_name']

after_2020['start_station_name'] = after_2020.progress_apply(start_fill_with_unique_id, axis=1)
after_2020['end_station_name'] = after_2020.progress_apply(end_fill_with_unique_id, axis=1)

100%|██████████| 24454526/24454526 [02:52<00:00, 141778.27it/s]


In [10]:
uknown_stations_data = [{'station_name': name, 'lat': lat, 'lng': lng}
        for (lat, lng), name in station_map.items()]

uknown_stations = pd.DataFrame(uknown_stations_data, columns=['station_name', 'lat', 'lng'])

In [11]:
stations = pd.concat([stations, uknown_stations]).drop_duplicates('station_name').reset_index(drop=True)

In [12]:
after_2020.to_parquet('..//data//after_2020.parquet', index=False)
before_2020.to_parquet('..//data//before_2020.parquet', index=False)
stations.to_csv('..//data//stations.csv', index=False)